# Lesson 2.2 — AWS EC2, S3 & CloudWatch Automation

**Level: Intermediate**

## Learning Objectives
- List EC2 instances with full **pagination** (never miss page 2+)
- Start, stop, and tag EC2 resources programmatically
- Manage S3: create, upload, download, lifecycle rules
- Publish custom metrics to CloudWatch from Python
- Describe and create CloudWatch alarms

**Produces:** `aws_resources.py` — EC2 / S3 / CloudWatch automation module.

*Official refs:*
- *[boto3 Paginators](https://docs.aws.amazon.com/boto3/latest/guide/paginators.html)*
- *[boto3 EC2 Reference](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/ec2.html)*
- *[boto3 S3 Reference](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)*
- *[boto3 CloudWatch Reference](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/cloudwatch.html)*

## Concepts

### Why Paginators?

Many AWS APIs return at most 1,000 items per request (e.g., `describe_instances`, `list_objects_v2`).
If you only call the API once, you silently miss all pages beyond the first.

From the [boto3 paginators guide](https://docs.aws.amazon.com/boto3/latest/guide/paginators.html):

> *"Paginators are a feature of boto3 that act as an abstraction over the process
> of iterating over an entire result set of a truncated API operation."*

```python
# BAD — only first page
response = ec2.describe_instances()

# GOOD — all pages automatically
paginator = ec2.get_paginator("describe_instances")
for page in paginator.paginate():
    for reservation in page["Reservations"]:
        ...
```

### Tags Are the Foundation of AWS Automation

Tags let you:
- **Filter**: find all instances with `Environment=production`
- **Cost-allocate**: AWS bills broken down by tag
- **Automate**: ops scripts that act on resources by tag

In [ ]:
import boto3, json, os
from datetime import datetime, timezone
from botocore.exceptions import ClientError

session = boto3.Session(region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))

# Simulated EC2 instances for demo (replace with real API calls)
SAMPLE_INSTANCES = [
    {"InstanceId": "i-0abc1234567", "State": {"Name": "running"},
     "InstanceType": "t3.micro",  "PrivateIpAddress": "10.0.1.10",
     "LaunchTime": datetime(2025, 1, 10, 8, 0, tzinfo=timezone.utc),
     "Tags": [{"Key": "Name", "Value": "web-server-1"},
               {"Key": "Environment", "Value": "production"}]},
    {"InstanceId": "i-0def5678901", "State": {"Name": "running"},
     "InstanceType": "t3.small",  "PrivateIpAddress": "10.0.1.11",
     "LaunchTime": datetime(2025, 1, 12, 9, 0, tzinfo=timezone.utc),
     "Tags": [{"Key": "Name", "Value": "worker-1"},
               {"Key": "Environment", "Value": "production"}]},
    {"InstanceId": "i-0ghi9012345", "State": {"Name": "stopped"},
     "InstanceType": "r6i.large",  "PrivateIpAddress": "10.0.2.5",
     "LaunchTime": datetime(2025, 1, 1,  0, 0, tzinfo=timezone.utc),
     "Tags": [{"Key": "Name", "Value": "db-backup"},
               {"Key": "Environment", "Value": "production"}]},
]
print(f"Loaded {len(SAMPLE_INSTANCES)} simulated instances.")

## Code-Along: Step 1 — EC2 Inventory with Pagination

In [ ]:
def parse_instance(inst: dict, region: str = "us-east-1") -> dict:
    """Normalise a raw EC2 instance dict into a flat record."""
    tags = {t["Key"]: t["Value"] for t in inst.get("Tags", [])}
    return {
        "instance_id": inst["InstanceId"],
        "name":        tags.get("Name", "(unnamed)"),
        "state":       inst["State"]["Name"],
        "type":        inst.get("InstanceType", ""),
        "private_ip":  inst.get("PrivateIpAddress", ""),
        "launch_time": inst.get("LaunchTime", ""),
        "region":      region,
        "tags":        tags,
    }

def list_ec2_instances(ec2_client, filters: list | None = None) -> list[dict]:
    """List all EC2 instances using paginators.

    From the boto3 paginators guide:
    https://docs.aws.amazon.com/boto3/latest/guide/paginators.html

    Uses get_paginator("describe_instances") so ALL pages are fetched
    regardless of how many instances exist in the account.
    """
    paginator = ec2_client.get_paginator("describe_instances")
    kwargs = {}
    if filters:
        kwargs["Filters"] = filters
    instances = []
    for page in paginator.paginate(**kwargs):
        for reservation in page["Reservations"]:
            for inst in reservation["Instances"]:
                instances.append(parse_instance(inst))
    return instances

# Demo with sample data
instances = [parse_instance(i) for i in SAMPLE_INSTANCES]
print(f"{'ID':<20} {'Name':<20} {'State':<10} {'Type':<12} {'Tags'}")
print("-" * 80)
for i in instances:
    tags = ", ".join(f"{k}={v}" for k, v in i['tags'].items())
    print(f"{i['instance_id']:<20} {i['name']:<20} {i['state']:<10} {i['type']:<12} {tags}")

## Code-Along: Step 2 — EC2 Lifecycle & Tagging

In [ ]:
def get_stopped_instances(instances: list[dict]) -> list[dict]:
    return [i for i in instances if i["state"] == "stopped"]

def tag_instances(ec2_client, instance_ids: list[str], tags: dict) -> None:
    """Apply tags to EC2 instances.

    Tags are the foundation of AWS cost allocation and automation.
    Reference: https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/Using_Tags.html
    """
    tag_list = [{"Key": k, "Value": v} for k, v in tags.items()]
    ec2_client.create_tags(Resources=instance_ids, Tags=tag_list)
    print(f"Tagged {len(instance_ids)} instance(s): {tags}")

stopped = get_stopped_instances(instances)
print(f"Stopped instances: {len(stopped)}")
for i in stopped:
    print(f"  {i['instance_id']} — {i['name']} (stopped since approx {i['launch_time']})")
print()
print("With real credentials, you would call:")
print("  stop_instances(ec2, ['i-0abc1234567'])")
print("  tag_instances(ec2, ['i-0abc1234567'], {'Patched': '2025-01-15', 'PatchedBy': 'ops-bot'})")

## Code-Along: Step 3 — S3 Operations

In [ ]:
def upload_json_to_s3(s3_client, bucket: str, key: str, data: dict) -> str:
    """Upload a JSON object to S3. Returns the s3:// URI.

    Reference: https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_object.html
    """
    body = json.dumps(data, indent=2, default=str).encode()
    s3_client.put_object(
        Bucket=bucket, Key=key, Body=body,
        ContentType="application/json",
    )
    return f"s3://{bucket}/{key}"

def list_s3_buckets(s3_client) -> list[dict]:
    """List all S3 buckets in the account."""
    response = s3_client.list_buckets()
    return [{"name": b["Name"], "created": b["CreationDate"].isoformat()}
            for b in response.get("Buckets", [])]

# Demonstrate the report structure that would be uploaded
report = {
    "scan_date": datetime.utcnow().isoformat() + "Z",
    "total_instances": len(instances),
    "stopped_count": len(get_stopped_instances(instances)),
    "instances": instances,
}
print("Sample report that would be uploaded to S3:")
print(json.dumps(report, indent=2, default=str)[:500] + "...")

## Code-Along: Step 4 — CloudWatch Custom Metrics

In [ ]:
def put_inventory_metrics(cw_client, instances: list[dict], region: str) -> None:
    """Publish EC2 inventory metrics to CloudWatch.

    Custom metrics allow you to track operational data that AWS does not
    measure natively (e.g., number of stopped instances per region).

    Reference: https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/publishingMetrics.html
    Namespace: OpsAutomation/Resources
    """
    now = datetime.now(timezone.utc)
    running = sum(1 for i in instances if i["state"] == "running")
    stopped = sum(1 for i in instances if i["state"] != "running")

    metric_data = [
        {"MetricName": "TotalInstances",   "Value": float(len(instances)),
         "Unit": "Count", "Timestamp": now,
         "Dimensions": [{"Name": "Region", "Value": region}]},
        {"MetricName": "RunningInstances", "Value": float(running),
         "Unit": "Count", "Timestamp": now,
         "Dimensions": [{"Name": "Region", "Value": region}]},
        {"MetricName": "StoppedInstances", "Value": float(stopped),
         "Unit": "Count", "Timestamp": now,
         "Dimensions": [{"Name": "Region", "Value": region}]},
    ]
    cw_client.put_metric_data(Namespace="OpsAutomation/Resources",
                              MetricData=metric_data)

print("CloudWatch metrics that WOULD be published (Namespace: OpsAutomation/Resources):")
for name, value in [("TotalInstances", 3), ("RunningInstances", 2), ("StoppedInstances", 1)]:
    print(f"  {name:<22} = {value}  [Region=us-east-1]")

In [ ]:
%%writefile aws_resources.py
"""AWS EC2, S3, and CloudWatch automation.

Reference:
  EC2:        https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/ec2.html
  S3:         https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html
  CloudWatch: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/cloudwatch.html
"""
import json
from datetime import datetime, timezone
from botocore.exceptions import ClientError

def list_ec2_instances(ec2_client, filters=None) -> list[dict]:
    """List EC2 instances using pagination (all pages)."""
    paginator = ec2_client.get_paginator("describe_instances")
    kwargs = {"Filters": filters} if filters else {}
    instances = []
    for page in paginator.paginate(**kwargs):
        for r in page["Reservations"]:
            for inst in r["Instances"]:
                tags = {t["Key"]: t["Value"] for t in inst.get("Tags", [])}
                instances.append({
                    "instance_id": inst["InstanceId"],
                    "name":        tags.get("Name", "(unnamed)"),
                    "state":       inst["State"]["Name"],
                    "type":        inst.get("InstanceType", ""),
                    "private_ip":  inst.get("PrivateIpAddress", ""),
                    "tags":        tags,
                })
    return instances

def upload_report_s3(s3_client, bucket: str, key: str, data: dict) -> str:
    """Upload a JSON report to S3. Returns the s3:// URI."""
    body = json.dumps(data, indent=2, default=str).encode()
    s3_client.put_object(Bucket=bucket, Key=key, Body=body,
                         ContentType="application/json")
    return f"s3://{bucket}/{key}"

def put_inventory_metrics(cw_client, instances: list[dict], region: str) -> None:
    """Publish EC2 inventory counts to CloudWatch."""
    now = datetime.now(timezone.utc)
    dims = [{"Name": "Region", "Value": region}]
    running = sum(1 for i in instances if i["state"] == "running")
    stopped = sum(1 for i in instances if i["state"] != "running")
    cw_client.put_metric_data(
        Namespace="OpsAutomation/Resources",
        MetricData=[
            {"MetricName": "TotalInstances",   "Value": float(len(instances)),
             "Unit": "Count", "Timestamp": now, "Dimensions": dims},
            {"MetricName": "RunningInstances", "Value": float(running),
             "Unit": "Count", "Timestamp": now, "Dimensions": dims},
            {"MetricName": "StoppedInstances", "Value": float(stopped),
             "Unit": "Count", "Timestamp": now, "Dimensions": dims},
        ]
    )


## Exercise

Write `find_untagged_instances(instances, required_tag="Environment") -> list[dict]`.

**Your task:**
1. Filter instances that are missing the `required_tag` key in their `tags` dict
2. Return `[{instance_id, name}]`
3. Test against `SAMPLE_INSTANCES` — add one instance with empty tags first

In [ ]:
# Add an untagged instance to the sample
untagged_sample = instances + [
    {"instance_id": "i-0mystery", "name": "mystery-box",
     "state": "running", "type": "t2.micro",
     "private_ip": "10.0.3.1", "region": "us-east-1", "tags": {}}
]

# TODO: implement find_untagged_instances
def find_untagged_instances(instances: list[dict], required_tag: str = "Environment") -> list[dict]:
    pass

untagged = find_untagged_instances(untagged_sample)
print(f"Untagged instances: {len(untagged)}")
for i in untagged:
    print(f"  {i['instance_id']} — {i.get('name','(unnamed)')}")


## Key Takeaways
- Always use `get_paginator()` for list operations — a single call only returns the first page
- Tags are the foundation of AWS automation: filter, group, cost-allocate, and automate by tag
- CloudWatch custom metrics allow you to push any operational data from Python scripts
- `datetime.now(timezone.utc)` — always use UTC for AWS timestamps

---
**Next:** [Lesson 2.3 — Azure SDK for Python](lesson-2.3-azure-sdk.ipynb)